# Learn the Massachusetts parser
Official saved report → extracted text/table → normalized DataFrame → validation → analysis.
This notebook runs offline and writes no files or SQLite. All money columns below are USD.

Category 1 is retail; Category 3 is online/mobile. `Total Online` is a printed control total, not an operator. Accrual Win and Taxable Gaming Revenue are separate measures.

Source: [Massachusetts Gaming Commission](https://massgaming.com/regulations/revenue/).


In [ ]:
from pathlib import Path
import sys
import hashlib
from datetime import datetime, timezone
from IPython.display import display

import pandas as pd
import pdfplumber

ROOT = Path.cwd()
if (ROOT / "gaming" / "src").is_dir():
    ROOT = ROOT / "gaming"
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.states import massachusetts as ma

pdf_path = ROOT / "tests" / "fixtures" / "MA" / "MGC-Revenue-Report-July-2026.pdf"
print(pdf_path)

## 1. Inspect the saved report before parsing
The source prints settled wagers, Accrual Win, Hold %, taxable revenue, and tax.
The excerpt below makes the source layout visible.

In [ ]:
with pdfplumber.open(pdf_path) as pdf:
    online_text = ma.find_online_operator_text(pdf)
print(online_text[:1800])
# Extracted lines preserve the native labels and layout before normalization.
display(pd.DataFrame({"extracted_source_line": online_text.splitlines()}))
source_operators, printed_total = ma.parse_online_section(online_text)
display(pd.DataFrame(source_operators).rename(columns={
    "wagers_settled": "Current & Future Wagers Settled (USD)",
    "accrual_win": "Accrual Win (USD)", "taxable_revenue": "Taxable Gaming Revenue (USD)",
    "tax_collected": "Tax Collected (USD)"}))


## 2. Call the same parser used by collection
`parse_revenue_pdf` checks the report period, finds online operators, and reconciles
all four money fields to Total Online. There is no second notebook-local parser.

In [ ]:
content = pdf_path.read_bytes()
operators, total_online, (year, month) = ma.parse_revenue_pdf(content)
parsed = pd.DataFrame(operators)
print(f"Reporting month: {year}-{month:02d}")
display(parsed)
# This timestamp is the in-memory demonstration time, not the original capture.
normalized = ma.build_normalized_rows(
    operators, year=year, month=month, total_online=total_online,
    source_url=ma.LANDING_URL, source_file=str(pdf_path.relative_to(ROOT)),
    source_sha256=hashlib.sha256(content).hexdigest(), retrieved_at=datetime.now(timezone.utc))
display(normalized[["operator", "row_type", "period_start", "period_end", "handle",
                    "gross_revenue", "taxable_revenue", "tax", "source_file", "source_sha256"]])


## 3. Show the reconciliation
The module defines the tolerances because the printed totals may have rounding differences.
Dropping an operator would cause this check to fail.

In [ ]:
_, total_online = ma.parse_online_section(online_text)
ma.reconcile_operators(operators, total_online)
fields = ["wagers_settled", "accrual_win", "taxable_revenue", "tax_collected"]
check = pd.DataFrame({"operator_sum": parsed[fields].sum(min_count=len(parsed)), "printed_total": pd.Series(total_online)})
check["difference"] = check["operator_sum"] - check["printed_total"]
check["allowed_difference"] = pd.Series(ma.RECONCILE_TOLERANCE)
check["missing_values"] = parsed[fields].isna().sum()
check["passed"] = check["difference"].abs().le(check["allowed_difference"]) & check["missing_values"].eq(0)
display(check)
assert check["passed"].all()
assert not normalized.duplicated(["period_start", "operator", "row_type"]).any()
print("The module reconciliation passed.")

## 4. Understand the stored columns
| Source column | SQLite column |
| --- | --- |
| Current & Future Wagers Settled | `handle` |
| Accrual Win by Licensee | `gross_revenue` |
| Taxable Gaming Revenue | `taxable_revenue` |
| Tax Collected | `tax` |

Accrual Win is already retained by the current parser. It does not replace taxable revenue.
This example proves the reconciliation for this file; it does not certify every historical row
or make Massachusetts economically identical to other states.

To study another layout, change `pdf_path` to `March-Rev-Report.pdf` in the same folder.
Use notebook 20 for collection and notebook 90 for analysis.

## 5. Analyze this one validated report
Accrual Win / settled wagers is reported hold. Shares use the printed total for the same month. This single report shows levels, not growth or a trend; use notebook 91 for the approved matched comparison. Missing denominators never become zeros.


In [ ]:
market = normalized[normalized.row_type.eq("official_statewide_total")].iloc[0]
analysis = normalized[normalized.row_type.eq("operator")].copy()
analysis["hold_pct"] = 100 * analysis.gross_revenue / analysis.handle.where(analysis.handle.gt(0))
analysis["handle_share_pct"] = 100 * analysis.handle / market.handle
analysis["accrual_win_share_pct"] = 100 * analysis.gross_revenue / market.gross_revenue
display(analysis[["operator", "handle", "gross_revenue", "hold_pct", "handle_share_pct",
                  "accrual_win_share_pct", "source_file"]])
